# `ingestion_AJGP.ipynb`
## The Art and Science of Selecting Appropriate Dressings for Acute Open Wounds in General Practice (AJGP 2022)
### Why this PDF needs a dedicated notebook
 
| Problem | Root cause | Fix |
|---|---|---|
| 3-column layout (left sidebar + 2 main columns) | Columns undetected by naive extraction | Explicit 3-column block classifier using x-centre thresholds |
| Tables (Table 1, Table 2) have no drawn borders | pdfplumber cannot auto-detect borderless tables | Hardcoded table data verified against raw block text |
| Section headers fused with first paragraph in same block | PDF layout compresses header + body | Match section openers with flexible regex |
| Boilerplate on every page (running header, footer, author block) | Journal reprint format | Strip known patterns during block filtering |
| Dressing prose (page 3 left+mid cols) should be combined with Table 2 | Different extraction sources | Post-process: merge prose descriptions with Table 2 rows |
 
**Strategy (produces ~19 clean chunks):**
1. PyMuPDF `get_text('blocks')` → 3-column aware extraction (left / mid / right / full)
2. Page-targeted narrative section extraction (Background, Introduction, Wound Dressings, etc.)
3. Hardcoded `TABLE1_DATA` (5 wound types) + `TABLE2_DATA` (6 dressing types) from Table 1 & Table 2
4. Hardcoded `DRESSING_PROSE` (section descriptions from page 3 left+mid columns)
5. Combine TABLE2_DATA rows with DRESSING_PROSE → 6 "one dressing one chunk" combined chunks
6. Optional LLM `ai_summary` enrichment
7. Export → `ingestion_output/AJGP_wound_dressings_kept.json`
8. Load to ChromaDB
 
### Expected chunk list (~19 chunks)
 
| # | Section | Source |
|---|---|---|
| 1 | Background & Objective | Page 1 left col |
| 2 | Introduction – Acute Wound Context | Page 1 mid+right cols |
| 3 | Wound Dressings – General Principles | Pages 1–2 |
| 4 | Description of Dressings – Overview | Page 2 right col |
| 5–9 | Table 1 – [5 wound types] | Hardcoded Table 1 |
| 10–15 | Dressing Types – [6 types] | Prose + Table 2 combined |
| 16 | Dressing Types and Costs | Page 3 right col |
| 17 | Role of Telemedicine | Page 3 right col + page 4 left col |
| 18 | Conclusion | Page 4 left col |
| 19 | Key Points | Page 4 left col |

In [43]:
# ── CELL 0 · Install dependencies (run once if needed) ────────────────────────
# !pip install pymupdf pdfplumber -q

In [57]:
# ── CELL 1 · Imports & configuration ──────────────────────────────────────────
import fitz          # PyMuPDF
import pdfplumber    # (used for inspection; tables extracted from hardcoded data)
import re
import json
import hashlib
import unicodedata
import statistics
from pathlib import Path
from collections import defaultdict, Counter
 
# ── paths ──────────────────────────────────────────────────────────────────────
PDF_PATH    = "../clinical_pdfs_v2/The art and science of selecting appropriate dressings for acute open wounds in general practice.pdf"
SOURCE_NAME = "The art and science of selecting appropriate dressings for acute open wounds in general practice.pdf"
OUT_DIR     = Path("../ingestion_output_no_ai")   # change to ingestion_output_ai or ingestion_output_no_ai as needed
OUT_DIR.mkdir(exist_ok=True)
 
# ── layout constants (measured from fitz block inspection) ────────────────────
PAGE_WIDTH     = 595.276   # A4 portrait
# 3-column boundaries (x-centre thresholds)
MID1_X         = 175.0     # left / mid column split
MID2_X         = 360.0     # mid / right column split
FULLWIDTH_FRAC = 0.50      # block width > 50% page → full-width element (skip in text flow)
 
# ── page indices (0-based) ─────────────────────────────────────────────────────
PAGE_1 = 0   # Title, Background, Objective, Discussion, Article intro start
PAGE_2 = 1   # Wound Dressings section + Table 1
PAGE_3 = 2   # Dressing types prose + Table 2 + Costs + Telemedicine start
PAGE_4 = 3   # Telemedicine conclusion + Conclusion + Key Points + Authors + References
 
# ── chunking limits ───────────────────────────────────────────────────────────
MAX_CHUNK_CHARS = 2400
MIN_CHUNK_CHARS = 80
 
print("✅  imports OK")
print(f"   PAGE_WIDTH={PAGE_WIDTH}pt  MID1={MID1_X}  MID2={MID2_X}  FULLWIDTH_FRAC={FULLWIDTH_FRAC}")

✅  imports OK
   PAGE_WIDTH=595.276pt  MID1=175.0  MID2=360.0  FULLWIDTH_FRAC=0.5


## Step 1 · Column-aware block extraction (PyMuPDF)
 
The AJGP layout has **3 columns**: a narrow left editorial sidebar, and two equal main text columns.
Measured x-centre thresholds: LEFT < 175 ≤ MID < 360 ≤ RIGHT.
Full-width elements (tables, page-wide rules) span > 50% of page width and are **skipped** in the
narrative text flow — table content is handled separately via hardcoded data.
 
Boilerplate blocks (running header, footer, author names) are filtered out by matching known text fragments.

In [58]:
# ── CELL 2 · Block extraction helpers ─────────────────────────────────────────
 
# Known boilerplate block texts to discard entirely (exact or partial match)
BOILERPLATE_EXACT = {
    "Focus\u2002 |\u2002 Clinical",
    "Focus | Clinical",
    "correspondence ajgp@racgp.org.au",
}
BOILERPLATE_STARTS = (
    "Reprinted from AJGP",
    "© The Royal Australian College",
    "The art and science of selecting appropriate dressings for acute open wounds in general practice",
    "Sankar N Sinha OAM",    # author affiliations block on page 4
    "Belinda Free RN",
    "Oliver Ladlow MBBS",
    "Competing interests:",
    "Funding:",
    "Provenance and peer review:",
    "Correspondence to:",
    "Sankar.sinha@utas.edu.au",
    "1.\tMohiuddin",          # references block starts
    "2.\tFranz MG",
)
BOILERPLATE_CONTAINS = (
    "Sankar N Sinha, Belinda Free",   # author names line on page 1
    "Authors\n",                       # "Authors" header on page 4
)
 
 
def is_boilerplate(text: str) -> bool:
    if text in BOILERPLATE_EXACT:
        return True
    for pat in BOILERPLATE_STARTS:
        if text.startswith(pat):
            return True
    for pat in BOILERPLATE_CONTAINS:
        if pat in text:
            return True
    # Page numbers (lone 3-digit numbers)
    if re.fullmatch(r"\d{3,4}", text.strip()):
        return True
    return False
 
 
def classify_col(x0: float, x1: float, width: float) -> str:
    """
    Return column classification for a block.
    full  → wider than FULLWIDTH_FRAC of page (table rows, header/footer rules)
    left  → x-centre < MID1_X
    mid   → MID1_X ≤ x-centre < MID2_X
    right → x-centre ≥ MID2_X
    """
    if width >= FULLWIDTH_FRAC * PAGE_WIDTH:
        return "full"
    x_ctr = (x0 + x1) / 2
    if x_ctr < MID1_X:
        return "left"
    elif x_ctr < MID2_X:
        return "mid"
    else:
        return "right"
 
 
def extract_page_blocks(pg: fitz.Page) -> list:
    """
    Return cleaned text blocks for one page.
    Each block: {x0, y0, x1, y1, text, col}
    - Image blocks, empty blocks and boilerplate blocks are discarded.
    - Full-width blocks (tables, rules) are tagged col='full'.
    """
    raw = pg.get_text("blocks", sort=False)
    blocks = []
    for b in raw:
        if b[6] != 0:          # skip non-text blocks
            continue
        text = unicodedata.normalize("NFKC", b[4]).strip()
        if not text:
            continue
        if is_boilerplate(text):
            continue
        x0, y0, x1, y1 = b[0], b[1], b[2], b[3]
        width = x1 - x0
        col = classify_col(x0, x1, width)
        blocks.append({"x0": x0, "y0": y0, "x1": x1, "y1": y1,
                        "text": text, "col": col})
    return blocks
 
 
def get_column_text(blocks: list, col: str, y0_min: float = 0, y0_max: float = 9999) -> str:
    """
    Extract and join text from a specific column, filtered by y0 range.
    Blocks are sorted by y0 (reading order within column).
    """
    filtered = [
        b for b in blocks
        if b["col"] == col and y0_min <= b["y0"] <= y0_max
    ]
    filtered.sort(key=lambda b: b["y0"])
    return "\n".join(b["text"] for b in filtered)
 
 
def make_chunk_id(source: str, section: str, idx: int = 0) -> str:
    raw = f"{source}::{section}::{idx}"
    return hashlib.md5(raw.encode()).hexdigest()[:12]
 
 
def split_into_chunks(text: str, max_chars: int = MAX_CHUNK_CHARS) -> list:
    """Split text into sub-chunks at paragraph/sentence boundaries."""
    if len(text) <= max_chars:
        return [text]
    chunks, remaining = [], text
    while len(remaining) > max_chars:
        split_pos = remaining.rfind("\n\n", 0, max_chars)
        if split_pos == -1:
            split_pos = remaining.rfind(". ", 0, max_chars)
        if split_pos == -1:
            split_pos = max_chars
        else:
            split_pos += 2
        chunks.append(remaining[:split_pos].strip())
        remaining = remaining[split_pos:].strip()
    if remaining.strip():
        chunks.append(remaining.strip())
    return [c for c in chunks if c]
 
 
def make_chunk(section: str, parent_section: str, text: str, chunk_index: int = 0) -> dict:
    return {
        "chunk_id":       make_chunk_id(SOURCE_NAME, section, chunk_index),
        "source":         SOURCE_NAME,
        "section":        section,
        "parent_section": parent_section,
        "chunk_index":    chunk_index,
        "char_count":     len(text),
        "text":           text,
        "ai_summary":     text,   # overwrite with LLM summary if ENABLE_AI_SUMMARY = True
    }
 
 
# ── Open PDF and extract all pages ────────────────────────────────────────────
doc = fitz.open(PDF_PATH)
print(f"PDF opened: {doc.page_count} pages, page width={doc[0].rect.width:.1f}pt")
 
page_blocks = {}
for pg_idx in range(doc.page_count):
    pg = doc[pg_idx]
    blocks = extract_page_blocks(pg)
    page_blocks[pg_idx] = blocks
    col_counts = Counter(b["col"] for b in blocks)
    print(f"  Page {pg_idx+1}: {len(blocks)} blocks  {dict(col_counts)}")

PDF opened: 4 pages, page width=595.3pt
  Page 1: 10 blocks  {'right': 3, 'left': 3, 'mid': 3, 'full': 1}
  Page 2: 28 blocks  {'full': 3, 'left': 5, 'mid': 10, 'right': 10}
  Page 3: 19 blocks  {'left': 3, 'mid': 5, 'right': 7, 'full': 4}
  Page 4: 32 blocks  {'full': 1, 'left': 4, 'mid': 16, 'right': 11}


## Step 2 · Extract narrative section texts
 
Each narrative section is extracted from specific page+column+y0 ranges.
This page-targeted approach is more reliable than full-document regex splitting for a 3-column layout.

In [59]:
# ── CELL 3 · Extract narrative section texts ──────────────────────────────────
 
def join_blocks(blocks, col, y0_min=0, y0_max=9999):
    """Collect and join text blocks from a column+y-range."""
    selected = [b for b in blocks
                if b["col"] == col and y0_min <= b["y0"] <= y0_max]
    selected.sort(key=lambda b: b["y0"])
    return "\n".join(b["text"] for b in selected)
 
# ── 1. Background & Objective ──────────────────────────────────────────────────
# Page 1 LEFT col: author names are at y0≈329 (skip by y0_min=360).
# Blocks at y0=368 (Background), 445 (Objective), 503 (Discussion).
text_background = join_blocks(page_blocks[PAGE_1], "left", y0_min=360)
print("─"*60)
print("Background & Objective:")
print(text_background[:600])
print("...")
print(text_background[-100:])
print(f"  → {len(text_background)} chars")
 
# ── 2. Introduction – Acute Wound Context ─────────────────────────────────────
# Page 1 MID col: y0≥328 (skip article title at y0≈87)
# Page 1 RIGHT col: only blocks BEFORE "Wound dressings" header (y0<550)
text_intro_mid   = join_blocks(page_blocks[PAGE_1], "mid",   y0_min=300)
text_intro_right = join_blocks(page_blocks[PAGE_1], "right", y0_max=549)
text_intro = (text_intro_mid + "\n" + text_intro_right).strip()
print("\nIntroduction – Acute Wound Context (first 400 chars):")
print(text_intro[:400])
print("...")
print(text_intro[-100:])
print(f"  → {len(text_intro)} chars")
 
# ── 3. Wound Dressings – General Principles ───────────────────────────────────
# Page 1 RIGHT col: "Wound dressings" section starts at y0≈556
# Page 2: LEFT + MID + RIGHT-early (y0≤163 on page 2 right col, before Description)
text_wd_p1_right = join_blocks(page_blocks[PAGE_1], "right", y0_min=550)
text_wd_p2_left  = join_blocks(page_blocks[PAGE_2], "left")
text_wd_p2_mid   = join_blocks(page_blocks[PAGE_2], "mid")
text_wd_p2_right = join_blocks(page_blocks[PAGE_2], "right", y0_max=163)
text_wound_dressings = "\n".join([
    text_wd_p1_right,
    text_wd_p2_left,
    text_wd_p2_mid,
    text_wd_p2_right,
]).strip()
print("\nWound Dressings – General Principles (first 400 chars):")
print(text_wound_dressings[:400])
print("...")
print(text_wound_dressings[-100:])
print(f"  → {len(text_wound_dressings)} chars")
 
# ── 4. Description of Dressings – Overview ────────────────────────────────────
# Page 2 RIGHT col: block at y0≈165 only (description intro paragraph)
text_description = join_blocks(page_blocks[PAGE_2], "right", y0_min=164, y0_max=380)
print("\nDescription of Dressings – Overview (first 400 chars):")
print(text_description[:400])
print("...")
print(text_description[-100:])
print(f"  → {len(text_description)} chars")
 
# ── 5. Dressing Types and Costs ───────────────────────────────────────────────
# Page 3 RIGHT col: blocks before "Role of telemedicine" (y0<320)
text_costs = join_blocks(page_blocks[PAGE_3], "right", y0_max=320)
print("\nDressing Types and Costs (first 400 chars):")
print(text_costs[:400])
print("...")
print(text_costs[-100:])
print(f"  → {len(text_costs)} chars")
 
# ── 6. Role of Telemedicine ───────────────────────────────────────────────────
# Page 3 RIGHT col: "Role of telemedicine" block (y0≥320)
# Page 4 LEFT col: telemedicine continuation + limitations (y0≤370 → before Conclusion)
# text_tele_p3 = join_blocks(page_blocks[PAGE_3], "right", y0_min=320)
# text_tele_p4 = join_blocks(page_blocks[PAGE_4], "left",  y0_max=370)
# text_telemedicine = (text_tele_p3 + "\n" + text_tele_p4).strip()
# print("\nRole of Telemedicine (first 400 chars):")
# print(text_telemedicine[:400])
# print("...")
# print(text_telemedicine[-100:])
# print(f"  → {len(text_telemedicine)} chars")
 
# ── 7. Conclusion ─────────────────────────────────────────────────────────────
# Page 4 LEFT col: y0≈372 to 530 (before Key points)
text_conclusion = join_blocks(page_blocks[PAGE_4], "left", y0_min=370, y0_max=530)
print("\nConclusion (first 400 chars):")
print(text_conclusion[:400])
print("...")
print(text_conclusion[-100:])
print(f"  → {len(text_conclusion)} chars")
 
# ── 8. Key Points ─────────────────────────────────────────────────────────────
# Page 4 LEFT col: y0≥530
text_keypoints = join_blocks(page_blocks[PAGE_4], "left", y0_min=530)
print("\nKey Points (first 400 chars):")
print(text_keypoints[:400])
print("...")
print(text_keypoints[-100:])
print(f"  → {len(text_keypoints)} chars")
 
print("\n✅  Narrative section texts extracted")

────────────────────────────────────────────────────────────
Background & Objective:
Background
Acute open wounds constitute a 
significant part of general practice. With 
an expanding global market of dressing 
products, selection of wound dressings 
remains an area of concern among 
doctors entering general practice.
Objective
The aim of this article is to describe a 
practical guide for choosing appropriate 
dressings when treating acute open 
wounds in general practice.
Discussion
Although dressing is an essential element 
of standard wound care, it is important to 
remember that dressing alone does not 
heal the wound. Judicious selection of 
dressings based on wound char
...
availability are 
important for delivering appropriate care 
towards timely healing of acute wounds.
  → 780 chars

Introduction – Acute Wound Context (first 400 chars):
A SUPERFICIAL OPEN WOUND with loss of 
epithelial lining is described as an ulcer. 
However, the words ‘open wound’ and 
‘ulcer’ are often u

## Step 3 · Table 1 — Wound categories with recommended dressings
 
**Why hardcoded?**  
Table 1 is rendered in the PDF without drawn cell borders. pdfplumber cannot auto-detect borderless tables,
and PyMuPDF's block extraction merges wound-type names with dressing text in the same block.
Hardcoded data matches the published table exactly and is verified against the raw block output below.
 
Each of the 5 wound types becomes one independent chunk.

In [60]:
# ── CELL 4 · Table 1 — hardcoded data + verification ─────────────────────────
 
TABLE1_DATA = [
    {
        "wound_type": "Skin tears",
        "dressings_recommended": (
            "Apply silicone-covered foam dressing directly over the wound.\n"
            "If bleeding, apply haemostatic alginate dressing as primary dressing "
            "under a silicone-coated foam dressing."
        ),
        "special_comments": (
            "Do not use any adhesive products on fragile skin as they may contribute "
            "to further skin tears, especially on forearms and hands of the elderly.\n"
            "Using a barrier wipe under the foam aids to secure application, reduce "
            "maceration and protect the skin on removal of the dressing.\n"
            "Remover wipes should also be used when removing a dressing from fragile skin.\n"
            "Removal of the dressing should be done in a direction that does not disturb "
            "viable tissue edges and flaps."
        ),
    },
    {
        "wound_type": "Minor cut / laceration",
        "dressings_recommended": (
            "Cover with a low-absorbent dressing that prevents further trauma and absorbs "
            "exudate (dry island dressing)."
        ),
        "special_comments": (
            "Check for diabetes and the presence of at least two signs or symptoms of "
            "inflammation (redness, warmth, induration, pain/tenderness) or purulent "
            "secretions indicating infection."
        ),
    },
    {
        "wound_type": "Postoperative wounds",
        "dressings_recommended": (
            "For wounds without exudate, dress over sutures with a film or thin hydrocolloid.\n"
            "For wounds with exudate, apply a bordered low-absorbent dressing (dry island dressing)."
        ),
        "special_comments": (
            "In case of wound dehiscence, organise prompt surgical review."
        ),
    },
    {
        "wound_type": "Small superficial burns",
        "dressings_recommended": (
            "After initial first aid treatment, cover burns area with hydrogel or "
            "hydrocolloid or film."
        ),
        "special_comments": (
            "Refer to burns specialist for burns that are deep or infected or located "
            "on hands, feet, face or genitalia."
        ),
    },
    {
        "wound_type": "Diabetic foot",
        "dressings_recommended": (
            "Apply a primary antimicrobial dressing product with secondary dressing "
            "according to exudate:\n"
            "  1. low exudate – low-absorbent pad\n"
            "  2. moderate exudate – silicone foam\n"
            "  3. high exudate – absorbent pad."
        ),
        "special_comments": (
            "Check pedal pulses and sensation; if there is poor perfusion, referral to a "
            "diabetic foot clinic or vascular surgeon is recommended.\n"
            "Silicone foams on feet, if applied, should be without borders and anchored "
            "with tape or bandages."
        ),
    },
]
 
 
def format_table1_chunk(row: dict) -> str:
    lines = [
        f"TABLE 1 — Wound Category: {row['wound_type']}",
        f"Source: AJGP 2022, Table 1. Wound categories with recommended dressings",
        "",
        f"Wound Type: {row['wound_type']}",
        "",
        "Dressings Recommended:",
        f"  {row['dressings_recommended']}",
        "",
        "Special Comments / Clinical Notes:",
        f"  {row['special_comments']}",
    ]
    return "\n".join(lines)
 
 
# Build chunk texts
table1_chunks: dict[str, str] = {}
for row in TABLE1_DATA:
    table1_chunks[row["wound_type"]] = format_table1_chunk(row)
 
print(f"Table 1 chunks built: {list(table1_chunks.keys())}")
 
# ── Verification: compare against raw block text from page 2 ─────────────────
print("\n── Raw Table 1 block text from page 2 (y0 ≥ 385) for verification ──")
tbl1_raw_blocks = [
    b for b in page_blocks[PAGE_2]
    if b["y0"] >= 385
]
tbl1_raw_blocks.sort(key=lambda b: (b["y0"], b["x0"]))
for b in tbl1_raw_blocks[:15]:
    print(f"  y0={b['y0']:.0f} col={b['col']:5s} x0={b['x0']:.0f}: {b['text'][:500].replace(chr(10),' ')!r}")
 
print("\n✅  Compare raw blocks above against TABLE1_DATA — adjust hardcoded data if needed.")

Table 1 chunks built: ['Skin tears', 'Minor cut / laceration', 'Postoperative wounds', 'Small superficial burns', 'Diabetic foot']

── Raw Table 1 block text from page 2 (y0 ≥ 385) for verification ──
  y0=386 col=left  x0=34: 'Table 1. Wound categories with recommended dressings'
  y0=404 col=full  x0=34: 'Wound type Dressings recommended Special comments'
  y0=421 col=mid   x0=34: 'Skin tears Apply silicone-covered foam dressing directly over  the wound.'
  y0=421 col=right x0=341: 'Do not use any adhesive products on fragile skin as  they may contribute to further skin tears, especially  on forearms and hands of the elderly.'
  y0=444 col=mid   x0=154: 'If bleeding, apply haemostatic alginate dressing  as primary dressing under a silicone-coated  foam dressing.'
  y0=453 col=right x0=341: 'Using a barrier wipe under the foam aids to secure  application, reduce maceration and protect the skin  on removal of the dressing.'
  y0=485 col=right x0=341: 'Remover wipes should also be used 

## Step 4 · Table 2 + Dressing Prose — Combined dressing type chunks
 
**Strategy:** Each dressing type gets **one unified chunk** combining:
- The structured Table 2 row (purpose/action, limitations, wear time)
- The prose description from the article body text (page 3 left+mid columns)
 
This is more useful for RAG than either alone:
the Table 2 provides structured facts (wear time, cautions)
while the prose provides clinical context and use cases.
 
**Table 2** data is hardcoded (same reason as Table 1 — borderless table).
**Dressing prose** is extracted automatically from the article text (page 3 left+mid columns)
and split by dressing-type bullet markers.

In [61]:
# ── CELL 5 · Table 2 hardcoded data ──────────────────────────────────────────
 
TABLE2_DATA = [
    {
        "dressing_class":       "Film Dressings",
        "table2_key":           "Films",         # key for matching prose
        "purpose_action": (
            "Permeable to gas but impermeable to bacteria and liquid. "
            "Useful on superficial wounds with minimum exudate."
        ),
        "limitations_cautions": "May be traumatic on removal.",
        "wear_time":            "1–4 days",
    },
    {
        "dressing_class":       "Foam Dressings",
        "table2_key":           "Foam",
        "purpose_action": (
            "Suitable for moderately exudating wounds, skin tears, "
            "skin grafts and donor sites."
        ),
        "limitations_cautions": (
            "Nonsilicone types should be avoided in patients with fragile skin."
        ),
        "wear_time":            "Up to seven days",
    },
    {
        "dressing_class":       "Low-Adherent / Low-Absorbent Dressings",
        "table2_key":           "Low adherence",
        "purpose_action": (
            "Passive breathable dressing for low-exudating wounds. "
            "Protection over sutures or shallow wounds."
        ),
        "limitations_cautions": (
            "Not suitable for fragile papery skin as adhesive border can cause skin tear on removal. "
            "Not showerproof. Require secondary dressings for absorbing exudate – added cost."
        ),
        "wear_time":            "1–4 days",
    },
    {
        "dressing_class":       "Hydrocolloid Dressings",
        "table2_key":           "Hydrocolloid",
        "purpose_action": (
            "The sheet form of the dressing is self-adhesive and waterproof, and it does "
            "not need a secondary dressing, which makes this dressing type easy to use."
        ),
        "limitations_cautions": "Low absorbency, produce unpleasant odour during removal.",
        "wear_time":            "Up to seven days",
    },
    {
        "dressing_class":       "Alginate Dressings",
        "table2_key":           "Alginate",
        "purpose_action": (
            "Promotes haemostasis in actively bleeding wounds, used in "
            "moderate-to-high-exudating wounds, wicks away fluid from the wound, "
            "can be used in packing wounds. Available in sheets or ropes."
        ),
        "limitations_cautions": (
            "Will dry firm within 48 hours; may need to be soaked off to remove. "
            "Allergic reaction has been reported."
        ),
        "wear_time":            "Up to two days",
    },
    {
        "dressing_class":       "Antimicrobial Dressings",
        "table2_key":           "Antimicrobial",
        "purpose_action": (
            "The clinical evidence supporting the routine use of antimicrobial "
            "dressings is weak."
        ),
        "limitations_cautions": (
            "Bacterial resistance with long-term use. "
            "High cost of silver-impregnated dressings."
        ),
        "wear_time":            "1–4 days",
    },
]
 
# ── Verification: raw Table 2 block text from page 3 ─────────────────────────
print("── Raw Table 2 blocks from page 3 (y0 ≥ 460) for verification ──")
tbl2_raw_blocks = [b for b in page_blocks[PAGE_3] if b["y0"] >= 460]
tbl2_raw_blocks.sort(key=lambda b: (b["y0"], b["x0"]))
for b in tbl2_raw_blocks:
    print(f"  y0={b['y0']:.0f} col={b['col']:5s} x0={b['x0']:.0f}: {b['text'][:80].replace(chr(10),' ')!r}")
 
print(f"\n✅  Compare raw blocks above against TABLE2_DATA — adjust if needed.")

── Raw Table 2 blocks from page 3 (y0 ≥ 460) for verification ──
  y0=466 col=left  x0=57: 'Table 2. Dressing types'
  y0=484 col=full  x0=57: 'Dressing class  (generic) Purpose/action Limitations and cautions Wear time'
  y0=511 col=full  x0=57: 'Films Permeable to gas but impermeable to bacteria and liquid.  Useful on superf'
  y0=538 col=full  x0=57: 'Foam Suitable for moderately exudating wounds, skin tears,  skin grafts and dono'
  y0=565 col=left  x0=57: 'Low adherence,  low-absorbent  dressing'
  y0=565 col=mid   x0=122: 'Passive breathable dressing for low-exudating wounds.'
  y0=565 col=right x0=334: 'Not suitable for fragile papery skin as adhesive  border can cause skin tear on '
  y0=565 col=right x0=510: '1–4 days'
  y0=578 col=mid   x0=122: 'Protection over sutures or shallow wounds.'
  y0=611 col=mid   x0=57: 'Hydrocolloid The sheet form of the dressing is self-adhesive and  waterproof, an'
  y0=611 col=right x0=334: 'Low absorbency, produce unpleasant odour  during remo

In [62]:
# ── CELL 6 · Extract dressing prose from page 3 columns + combine with Table 2 ─
 
# The prose descriptions appear in page 3 LEFT + MID columns (before Table 2 area)
prose_raw_left = join_blocks(page_blocks[PAGE_3], "left",  y0_max=460)
prose_raw_mid  = join_blocks(page_blocks[PAGE_3], "mid",   y0_max=460)
dressing_prose_raw = prose_raw_left + "\n" + prose_raw_mid
print("Raw dressing prose (first 600 chars):")
print(dressing_prose_raw[:600])
print()
 
# ── Parse individual dressing prose blocks ─────────────────────────────────────
# Bullet markers (as they appear after NFKC normalisation):
# Use \s+ to handle any combination of spaces or tabs (\t) 
# and ensure the plural 's' is handled.
DRESSING_PROSE_PATTERNS = [
    ("Films",          r"Film\s+dressings\s+[\u2013\u2014\-]"),
    ("Foam",           r"Foam\s+dressings\s+[\u2013\u2014\-]"),
    ("Low adherence",  r"Low-adherent(?:,\s*low-absorbent)?\s+dressings\s+[\u2013\u2014\-]"),
    ("Hydrocolloid",   r"Hydrocolloid\s+dressings\s+[\u2013\u2014\-]"),
    ("Alginate",       r"Alginate\s+dressings\s+[\u2013\u2014\-]"),
    ("Antimicrobial",  r"Antimicrobial(?:[^\n]*?)dressings\s+[\u2013\u2014\-]"),
]
 
# Find split positions
prose_splits = []
for key, pattern in DRESSING_PROSE_PATTERNS:
    m = re.search(pattern, dressing_prose_raw, re.IGNORECASE)
    if m:
        prose_splits.append((m.start(), key))
    else:
        print(f"  ⚠️  Pattern not found for {key!r} — check dressing_prose_raw above")
 
prose_splits.sort(key=lambda s: s[0])
print(f"Prose splits found: {[(k, pos) for pos, k in prose_splits]}")
 
# Slice out each dressing's prose
dressing_prose: dict[str, str] = {}
for i, (pos, key) in enumerate(prose_splits):
    end_pos = prose_splits[i+1][0] if i+1 < len(prose_splits) else len(dressing_prose_raw)
    raw_prose = dressing_prose_raw[pos:end_pos].strip()
    # Clean: remove the bullet "•\t" prefix if present
    raw_prose = re.sub(r"^[•\t\s]+", "", raw_prose)
    dressing_prose[key] = raw_prose
 
print("\nExtracted prose per dressing type:")
for key, prose in dressing_prose.items():
    print(f"  {key!r}: {len(prose)} chars — {prose[:80].replace(chr(10),' ')!r}")
 
# ── Build combined chunk text for each dressing type ──────────────────────────
def format_dressing_combined_chunk(t2_row: dict, prose_text: str) -> str:
    prose_block = prose_text if prose_text else "(See Table 2 structured data.)"
    lines = [
        f"Dressing Type: {t2_row['dressing_class']}",
        f"Source: AJGP 2022 — Section description + Table 2. Dressing types",
        "",
        "PROPERTIES (Table 2):",
        f"  Purpose/Action: {t2_row['purpose_action']}",
        f"  Limitations & Cautions: {t2_row['limitations_cautions']}",
        f"  Wear Time: {t2_row['wear_time']}",
        "",
        "DETAILED DESCRIPTION (section text):",
        f"  {prose_block}",
    ]
    return "\n".join(lines)
 
 
dressing_combined_chunks: dict[str, str] = {}
for row in TABLE2_DATA:
    key = row["table2_key"]
    prose = dressing_prose.get(key, "")
    dressing_combined_chunks[row["dressing_class"]] = format_dressing_combined_chunk(row, prose)
 
print("\nCombined dressing chunks built:")
for name, text in dressing_combined_chunks.items():
    print(f"  {name!r}: {len(text)} chars")
print("\n✅  Dressing chunks ready")

Raw dressing prose (first 600 chars):
Dressing types include the following:
•	 Film dressings – these materials are 
semipermeable and demonstrate 
beneficial effect in the healing of 
superficial burns, minor abrasions 
and lacerations. Many now include a 
skin-safe adhesive to reduce the risk 
of trauma in fragile skin; however, 
caution should be taken if the 
patient has particularly vulnerable 
skin. It may be advisable to use a 
skin protectant (barrier) product 
underneath the dressing to avoid any 
harm. Film dressings are most useful 
for postoperative wounds healing by 
primary intention as they facilitate easy 
monitori

Prose splits found: [('Films', 41), ('Foam', 622), ('Low adherence', 1061), ('Hydrocolloid', 1281), ('Alginate', 1697), ('Antimicrobial', 1967)]

Extracted prose per dressing type:
  'Films': 579 chars — 'Film dressings – these materials are  semipermeable and demonstrate  beneficial '
  'Foam': 437 chars — 'Foam dressings – these are film  dressings with th

## Step 5 · Assemble final chunk list

In [63]:
# ── CELL 7 · Assemble all chunks ──────────────────────────────────────────────
 
chunks: list[dict] = []
 
 
def add_chunk(section, parent, text, idx=0):
    if len(text) < MIN_CHUNK_CHARS:
        print(f"  ⚠️  Skipping {section!r} — too short ({len(text)} chars)")
        return
    for ci, chunk_text in enumerate(split_into_chunks(text)):
        if len(chunk_text) < MIN_CHUNK_CHARS:
            continue
        chunks.append(make_chunk(section, parent, chunk_text, chunk_index=idx + ci))
 
 
# ── 1. Background & Objective ────────────────────────────────────────────────
add_chunk("Background & Objective",
          "Article Context",
          text_background)
 
# ── 2. Introduction – Acute Wound Context ─────────────────────────────────────
add_chunk("Introduction – Acute Wound Context",
          "Article Context",
          text_intro)
 
# ── 3. Wound Dressings – General Principles ───────────────────────────────────
add_chunk("Wound Dressings – General Principles",
          "Wound Dressings",
          text_wound_dressings)
 
# ── 4. Description of Dressings – Overview ────────────────────────────────────
add_chunk("Description of Dressings – Overview",
          "Wound Dressings",
          text_description)
 
# ── 5–9. Table 1 — one chunk per wound type ───────────────────────────────────
for row in TABLE1_DATA:
    text = table1_chunks[row["wound_type"]]
    add_chunk(f"Table 1 – {row['wound_type']}",
              "Table 1: Wound Categories",
              text)
 
# ── 10–15. Dressing type combined chunks ──────────────────────────────────────
for row in TABLE2_DATA:
    text = dressing_combined_chunks[row["dressing_class"]]
    add_chunk(f"Dressing Types – {row['dressing_class']}",
              "Dressing Types",
              text)
 
# ── 16. Dressing Types and Costs ──────────────────────────────────────────────
add_chunk("Dressing Types and Costs",
          "Clinical Practice Considerations",
          text_costs)
 
# ── 17. Role of Telemedicine ──────────────────────────────────────────────────
# add_chunk("Role of Telemedicine",
#           "Clinical Practice Considerations",
#           text_telemedicine)
 
# ── 18. Conclusion ────────────────────────────────────────────────────────────
add_chunk("Conclusion",
          "Conclusion",
          text_conclusion)
 
# ── 19. Key Points ────────────────────────────────────────────────────────────
add_chunk("Key Points",
          "Conclusion",
          text_keypoints)
 
 
print(f"\nTotal chunks assembled: {len(chunks)}")
print(f"  Expected: ~19")
print()
for i, c in enumerate(chunks, 1):
    print(f"  [{i:2d}] {c['section'][:55]!r:57s}  chars={c['char_count']}")


Total chunks assembled: 19
  Expected: ~19

  [ 1] 'Background & Objective'                                   chars=780
  [ 2] 'Introduction – Acute Wound Context'                       chars=2092
  [ 3] 'Wound Dressings – General Principles'                     chars=1962
  [ 4] 'Wound Dressings – General Principles'                     chars=1324
  [ 5] 'Description of Dressings – Overview'                      chars=663
  [ 6] 'Table 1 – Skin tears'                                     chars=826
  [ 7] 'Table 1 – Minor cut / laceration'                         chars=506
  [ 8] 'Table 1 – Postoperative wounds'                           chars=447
  [ 9] 'Table 1 – Small superficial burns'                        chars=421
  [10] 'Table 1 – Diabetic foot'                                  chars=636
  [11] 'Dressing Types – Film Dressings'                          chars=941
  [12] 'Dressing Types – Foam Dressings'                          chars=820
  [13] 'Dressing Types – Low-Adherent / 

## Step 6 · Quality validation

In [64]:
# ── CELL 8 · Quality checks ──────────────────────────────────────────────────
 
all_combined = " ".join(c["text"].lower() for c in chunks)
 
MUST_CONTAIN = [
    ("hydrocolloid",             "Hydrocolloid dressing"),
    ("alginate",                 "Alginate dressing"),
    ("foam dressing",            "Foam dressing"),
    ("antimicrobial",            "Antimicrobial dressing"),
    ("film dressing",            "Film dressing"),
    ("low-adherent",             "Low-adherent dressing"),
    ("silver",                   "Silver mention (antimicrobial)"),
    ("iodine",                   "Iodine mention (antimicrobial)"),
    ("skin tears",               "Skin tears wound type"),
    ("diabetic foot",            "Diabetic foot wound type"),
    ("postoperative",            "Postoperative wound type"),
    ("silicone foam",            "Silicone foam dressing"),
    ("wear time",                "Wear time info"),
    ("fragile skin",             "Fragile skin warning"),
    ("haemostasis",              "Haemostasis (alginate property)"),
    ("telehealth",               "Telehealth/telemedicine"),
    ("cost",                     "Cost consideration"),
    ("conclusion",               "Conclusion section"),
    ("key points",               "Key points section"),
    ("moist environment",        "Moist wound healing principle"),
]
 
print("=" * 70)
print("CHUNK QUALITY REPORT — AJGP 2022 Wound Dressings")
print("=" * 70)
 
char_counts = [c["char_count"] for c in chunks]
print(f"\n📦 Total chunks     : {len(chunks)}")
print(f"   Chars — min   : {min(char_counts)}")
print(f"   Chars — mean  : {statistics.mean(char_counts):.0f}")
print(f"   Chars — max   : {max(char_counts)}")
oversized = [c for c in chunks if c["char_count"] > MAX_CHUNK_CHARS]
print(f"   Oversized (>{MAX_CHUNK_CHARS}): {len(oversized)}"
      + (" ← check split logic" if oversized else ""))
 
# Duplicate check
seen, dupes = {}, []
for c in chunks:
    key = c["text"][:200]
    if key in seen:
        dupes.append((seen[key], c["chunk_id"]))
    else:
        seen[key] = c["chunk_id"]
print(f"\n🔁 Duplicates       : {len(dupes)}"
      + (" ← check assembly" if dupes else " ✓"))
 
# Clinical coverage
print("\n✅ Clinical content coverage:")
all_ok = True
for keyword, label in MUST_CONTAIN:
    found = keyword.lower() in all_combined
    status = "  ✅" if found else "  ❌ MISSING"
    if not found:
        all_ok = False
    print(f"{status}  {label!r}")
 
# Sections
print("\n🗂  Chunks by parent section:")
by_parent = Counter(c["parent_section"] for c in chunks)
for parent, cnt in sorted(by_parent.items()):
    print(f"  {cnt:2d} × {parent!r}")
 
if all_ok and not dupes and not oversized:
    print("\n✅  All checks passed — pipeline output is clean.")
else:
    print("\n⚠️  One or more checks failed — review above.")

CHUNK QUALITY REPORT — AJGP 2022 Wound Dressings

📦 Total chunks     : 19
   Chars — min   : 421
   Chars — mean  : 852
   Chars — max   : 2092
   Oversized (>2400): 0

🔁 Duplicates       : 0 ✓

✅ Clinical content coverage:
  ✅  'Hydrocolloid dressing'
  ✅  'Alginate dressing'
  ✅  'Foam dressing'
  ✅  'Antimicrobial dressing'
  ✅  'Film dressing'
  ✅  'Low-adherent dressing'
  ✅  'Silver mention (antimicrobial)'
  ✅  'Iodine mention (antimicrobial)'
  ✅  'Skin tears wound type'
  ✅  'Diabetic foot wound type'
  ✅  'Postoperative wound type'
  ✅  'Silicone foam dressing'
  ✅  'Wear time info'
  ✅  'Fragile skin warning'
  ✅  'Haemostasis (alginate property)'
  ✅  'Telehealth/telemedicine'
  ✅  'Cost consideration'
  ✅  'Conclusion section'
  ✅  'Key points section'
  ✅  'Moist wound healing principle'

🗂  Chunks by parent section:
   2 × 'Article Context'
   1 × 'Clinical Practice Considerations'
   2 × 'Conclusion'
   6 × 'Dressing Types'
   5 × 'Table 1: Wound Categories'
   3 × 'Wou

In [65]:
# ── CELL 9 · Spot-check individual chunks ────────────────────────────────────
 
def preview_chunk(identifier):
    """Preview by list index (int) or section name substring (str)."""
    if isinstance(identifier, int):
        c = chunks[identifier]
    else:
        matches = [x for x in chunks if identifier.lower() in x["section"].lower()]
        if not matches:
            print(f"No chunk matching {identifier!r}")
            return
        c = matches[0]
    print(f"\n{'─'*65}")
    print(f"chunk_id   : {c['chunk_id']}")
    print(f"section    : {c['section']}")
    print(f"parent     : {c['parent_section']}")
    print(f"chars      : {c['char_count']}")
    print(f"TEXT:")
    print(c["text"][:700])
    if len(c["text"]) > 700:
        print("... [truncated]")
 
for key in ["Background", "Alginate", "Table 1 – Skin", "Table 1 – Diabetic",
            "Dressing Types – Film", "Dressing Types – Antimicrobial", "Conclusion"]:
    preview_chunk(key)


─────────────────────────────────────────────────────────────────
chunk_id   : 65414974ff6f
section    : Background & Objective
parent     : Article Context
chars      : 780
TEXT:
Background
Acute open wounds constitute a 
significant part of general practice. With 
an expanding global market of dressing 
products, selection of wound dressings 
remains an area of concern among 
doctors entering general practice.
Objective
The aim of this article is to describe a 
practical guide for choosing appropriate 
dressings when treating acute open 
wounds in general practice.
Discussion
Although dressing is an essential element 
of standard wound care, it is important to 
remember that dressing alone does not 
heal the wound. Judicious selection of 
dressings based on wound characteristics, 
physical properties of dressings and their 
costs, shelf life and availability are 
im
... [truncated]

─────────────────────────────────────────────────────────────────
chunk_id   : 38f855618fe2
section  

## Step 7 · (Optional) LLM `ai_summary` enrichment
 
Set `ENABLE_AI_SUMMARY = True` to replace raw extracted text in `ai_summary` with an LLM-generated clinical summary.
 
`ai_summary` is what gets stored as `page_content` in ChromaDB and used as `reference_contexts` in RAGAS testsets.  
If you skip this step, `ai_summary == text` (raw extraction), which is already clean and readable.

In [66]:
# ── CELL 10 · LLM ai_summary enrichment (optional) ──────────────────────────
ENABLE_AI_SUMMARY = False   # ← set True when OPENAI_API_KEY is available
 
# Sections that benefit most from LLM summarisation (structured table data)
AI_PRIORITY_SECTIONS = {
    s["section"] for s in chunks
    if any(kw in s["section"] for kw in ["Table 1", "Dressing Types"])
}
 
SYSTEM_PROMPT = (
    "You are a clinical wound-care summarisation assistant. "
    "Rewrite the following wound-dressing text as a clear, complete, self-contained "
    "clinical summary for a retrieval-augmented generation system. "
    "Preserve ALL clinical facts: dressing names, wound types, indications, "
    "contraindications, wear time, special comments, and safety warnings. "
    "Return only the summary — no preamble, no commentary."
)
 
if ENABLE_AI_SUMMARY:
    import os
    from openai import OpenAI
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
 
    def get_ai_summary(text: str) -> str:
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            temperature=0,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": text},
            ],
        )
        return resp.choices[0].message.content.strip()
 
    print(f"Running AI summaries for {len(chunks)} chunks...")
    for i, c in enumerate(chunks):
        note = " (priority)" if c["section"] in AI_PRIORITY_SECTIONS else ""
        print(f"  [{i+1:2d}/{len(chunks)}] {c['section']}{note}")
        c["ai_summary"] = get_ai_summary(c["text"])
    print("\n✅  AI summaries complete")
 
else:
    ai_count = sum(1 for c in chunks if c["ai_summary"] != c["text"])
    print("ℹ️ AI summary disabled — ai_summary == text (raw extraction)")
    print(f"   {ai_count} chunks have a different ai_summary from a previous run")
    print("   Set ENABLE_AI_SUMMARY = True to regenerate.")

ℹ️ AI summary disabled — ai_summary == text (raw extraction)
   0 chunks have a different ai_summary from a previous run
   Set ENABLE_AI_SUMMARY = True to regenerate.


## Step 8 · Export JSON

In [67]:
# ── CELL 11 · Export ChromaDB-ready JSON ─────────────────────────────────────
 
output = {
    "meta": {
        "total_chunks":  len(chunks),
        "kept_count":    len(chunks),
        "ai_summarised": sum(1 for c in chunks if c["ai_summary"] != c["text"]),
        "extraction":    "PyMuPDF native text blocks (3-column aware); Table 1 + Table 2 hardcoded from PDF",
        "chunking":      "page-targeted section extraction; one dressing one chunk (prose + Table 2 combined)",
        "pages_used":    [1, 2, 3, 4],
        "tables_used":   ["Table 1 (Wound categories)", "Table 2 (Dressing types)"],
        "chunk_params": {
            "max_characters": MAX_CHUNK_CHARS,
            "min_characters": MIN_CHUNK_CHARS,
        },
        "note": (
            "Use ai_summary field for reference_contexts in RAGAS testset "
            "and as page_content in ChromaDB. ai_summary == text when "
            "ENABLE_AI_SUMMARY=False."
        ),
    },
    "kept_ids_by_source": {
        SOURCE_NAME: [c["chunk_id"] for c in chunks]
    },
    "kept_chunks": [
        {
            "chunk_id":       c["chunk_id"],
            "source":         c["source"],
            "section":        c["section"],
            "parent_section": c["parent_section"],
            "chunk_index":    c["chunk_index"],
            "char_count":     c["char_count"],
            "text":           c["text"],
            "ai_summary":     c["ai_summary"],
        }
        for c in chunks
    ],
}
 
out_path = OUT_DIR / "AJGP_wound_dressings_kept.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)
 
print(f"✅  Exported {len(chunks)} chunks → {out_path}")
print(f"    File size: {out_path.stat().st_size / 1024:.1f} KB")

✅  Exported 19 chunks → ..\ingestion_output_no_ai\AJGP_wound_dressings_kept.json
    File size: 40.7 KB
